# 2. Tính toán chỉ số đánh giá Khuyến nghị việc làm (KB4)
Notebook này đọc dữ liệu đối chiếu và tính toán các chỉ số: **Precision@K (K=1, 3, 5)** và **MRR**.


## Bước 2.1: Nạp tập dữ liệu đối chứng


In [4]:
import json
from pathlib import Path

workspace_root = Path(r'f:\HCMUS_KH\LuanVan\JobVisualization_BE')
script_dir = workspace_root / 'KiemThu' / 'MatchingCV'

with open(script_dir / 'ground_truth_matching.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
print(f'Nạp thành công dữ liệu đối chiếu của {len(dataset)} CVs.')


Nạp thành công dữ liệu đối chiếu của 10 CVs.


## Bước 2.2: Tính toán Precision@K và MRR


In [5]:
import numpy as np

cv_results = []
p1_list = []
p3_list = []
p5_list = []
rr_list = []

for cv_item in dataset:
    recs = cv_item['recommendations']
    relevance = [1 if r['true_label'] == 'RELEVANT' else 0 for r in recs]
    
    # Precision@K
    p1 = relevance[0] if len(relevance) >= 1 else 0.0
    p3 = sum(relevance[:3]) / min(3, len(relevance)) if len(relevance) > 0 else 0.0
    p5 = sum(relevance[:5]) / min(5, len(relevance)) if len(relevance) > 0 else 0.0
    
    p1_list.append(p1)
    p3_list.append(p3)
    p5_list.append(p5)
    
    # Reciprocal Rank (RR) cho MRR
    rr = 0.0
    for idx, rel in enumerate(relevance, 1):
        if rel == 1:
            rr = 1.0 / idx
            break
    rr_list.append(rr)
    
    cv_results.append({
        'cv_file': cv_item['cv_file'],
        'name': cv_item['candidate_name'],
        'p1': p1,
        'p3': p3,
        'p5': p5,
        'rr': rr
    })

mean_p1 = np.mean(p1_list)
mean_p3 = np.mean(p3_list)
mean_p5 = np.mean(p5_list)
mean_mrr = np.mean(rr_list)

print('=== CHỈ SỐ ĐÁNH GIÁ CHUNG TOÀN HỆ THỐNG ===')
print(f'  Mean Precision@1 (P@1) : {mean_p1*100:.2f}%')
print(f'  Mean Precision@3 (P@3) : {mean_p3*100:.2f}%')
print(f'  Mean Precision@5 (P@5) : {mean_p5*100:.2f}%')
print(f'  Mean Reciprocal Rank (MRR) : {mean_mrr:.4f}')
print('='*50)


=== CHỈ SỐ ĐÁNH GIÁ CHUNG TOÀN HỆ THỐNG ===
  Mean Precision@1 (P@1) : 60.00%
  Mean Precision@3 (P@3) : 55.00%
  Mean Precision@5 (P@5) : 55.00%
  Mean Reciprocal Rank (MRR) : 0.6000


## Bước 2.3: In bảng đối soát chi tiết (cho báo cáo Luận văn)


In [6]:
print('### Bảng đối soát chi tiết chỉ số gợi ý việc làm của từng ứng viên:')
print('| STT | Họ tên ứng viên | Tên File CV | Precision@1 | Precision@3 | Precision@5 | Reciprocal Rank (RR) |')
print('|---|---|---|---|---|---|---|')
for idx, r in enumerate(cv_results, 1):
    print(f'| {idx} | {r["name"]} | {r["cv_file"]} | {r["p1"]*100:.1f}% | {r["p3"]*100:.1f}% | {r["p5"]*100:.1f}% | {r["rr"]:.4f} |')

print('\n\n### Chi tiết từng cặp so khớp khuyến nghị:')
print('| CV | Hạng | Công ty | Tiêu đề Job | Điểm số | Dự đoán | Nhãn chuẩn | Trạng thái |')
print('|---|---|---|---|---|---|---|---|')
for cv_item in dataset:
    cv_file = cv_item['cv_file']
    for idx, r in enumerate(cv_item['recommendations'], 1):
        pred = r['predicted_label']
        true = r['true_label']
        status = 'TP (Gợi ý đúng)' if pred == 'RELEVANT' and true == 'RELEVANT' else \
                 'TN (Lọc đúng)' if pred == 'IRRELEVANT' and true == 'IRRELEVANT' else \
                 'FP (Gợi ý sai)' if pred == 'RELEVANT' and true == 'IRRELEVANT' else \
                 'FN (Bỏ sót)'
        
        print(f'| {cv_file} | {idx} | {r["company_name"]} | {r["job_title"]} | {r["match_score"]}% | {pred} | {true} | {status} |')


### Bảng đối soát chi tiết chỉ số gợi ý việc làm của từng ứng viên:
| STT | Họ tên ứng viên | Tên File CV | Precision@1 | Precision@3 | Precision@5 | Reciprocal Rank (RR) |
|---|---|---|---|---|---|---|
| 1 | ý Trần | 1783877254669-TruongNguyenNgocMinh.pdf | 100.0% | 100.0% | 100.0% | 1.0000 |
| 2 | ý Trần | 1781802615621-TranThiMyY_CV_Binance_BAP_Product_Manager.pdf | 100.0% | 50.0% | 50.0% | 1.0000 |
| 3 | Nhật Nguyễn | 1782626199971-CV_nguyendinhminhnhat.pdf | 0.0% | 0.0% | 0.0% | 0.0000 |
| 4 | ý Trần | 1782052362905-CV2_Web_Developer.pdf | 100.0% | 100.0% | 100.0% | 1.0000 |
| 5 | Loc Nguyen | 1782646920464-NguyenTanLoc_CV.pdf | 0.0% | 0.0% | 0.0% | 0.0000 |
| 6 | Huyen Thai | CV_ATTT_DQD.jpg | 0.0% | 0.0% | 0.0% | 0.0000 |
| 7 | Huyền Thái | 1782576938348-CV_ATTT_DQD.jpg | 0.0% | 0.0% | 0.0% | 0.0000 |
| 8 | Thái Thị Kim Huyền | ITBA_ThaiThiKimHuyen_CV.pdf | 100.0% | 100.0% | 100.0% | 1.0000 |
| 9 | Loc Nguyen | 1782646694903-NguyenTanLoc_CV.pdf | 100.0% | 100.0% | 100.0% | 1.000